In [80]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from torchsummary import summary

from tqdm import tqdm
device = "cuda" if torch.cuda.is_available() else "cpu"

In [81]:
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor()
])

In [82]:
train_datasets = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)

In [83]:
from torch.utils.data import Subset
train_size = 10000
np.random.seed(42)

indices = np.random.choice(len(train_datasets), train_size, replace=False)
np.random.seed()

train_sub_dataset = Subset(train_datasets, indices)

In [84]:
train_dataloader = DataLoader(train_sub_dataset, batch_size=64, shuffle=True)
eval_dataloader = DataLoader(train_sub_dataset, batch_size=len(train_sub_dataset), shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=len(test_dataset), shuffle=False)

In [85]:
class ResnetBlock(nn.Module):
    def __init__(self, in_c,out_c, stride):
        super().__init__()

        self.main_path = nn.Sequential(
            nn.Conv2d(in_c, out_c, kernel_size=3, padding=1, stride=stride),
            nn.BatchNorm2d(out_c),
            nn.ReLU(True),

            nn.Conv2d(out_c, out_c, kernel_size=3, padding=1, stride=1),
            nn.BatchNorm2d(out_c),
        )
        if stride != 1 or in_c != out_c:
            self.shortcut =nn.Sequential(
                nn.Conv2d(in_c, out_c, kernel_size=1, stride=stride)
            )
        else:
            self.shortcut = nn.Identity()
    def forward(self, x):
        return nn.ReLU()(self.main_path(x) + self.shortcut(x))

In [86]:
class ResNet(nn.Module):
    def __init__(self, num_class=10):
        super().__init__()
        self.extractor = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            ResnetBlock(64, 64, 2),
            ResnetBlock(64, 64, 1),


            ResnetBlock(64, 128, 2),
            ResnetBlock(128, 128, 1),


            ResnetBlock(128, 256, 2),
            ResnetBlock(256, 256, 1),

            ResnetBlock(256, 512, 2),
            ResnetBlock(512, 512, 1),

            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 256),
            nn.Dropout(0.5),
            nn.ReLU(True),
            nn.Linear(256, num_class)
        )

    def forward(self, x):
        x = self.extractor(x)
        x = self.fc(x)
        return x


In [87]:
model = ResNet().to(device)

In [88]:
criterion = nn.CrossEntropyLoss(reduction="sum")
optimizer = optim.AdamW(model.parameters(), lr=1e-3)


num_epochs = 5

for epoch in range(num_epochs):
    # Train the model
    model.train()
    epoch_loss = 0 # Total loss over all training samples
    for images, labels in tqdm(train_dataloader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)
        epoch_loss += loss.item()

        loss.backward()
        optimizer.step()

    # Evaluation
    model.eval()
    with torch.no_grad():
        for images, labels in eval_dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)

        predictions = torch.argmax(outputs, dim=1)
        epoch_correct_prob = torch.mean((predictions == labels).float()).item()

    print(f"Loss: {epoch_loss / len(train_sub_dataset)}")
    print(f"Accuracy: {epoch_correct_prob * 100}%")

100%|██████████| 157/157 [04:55<00:00,  1.88s/it]


Loss: 0.373678143787384
Accuracy: 94.1100001335144%


100%|██████████| 157/157 [04:35<00:00,  1.76s/it]


Loss: 0.1386742325425148
Accuracy: 95.8899974822998%


100%|██████████| 157/157 [04:41<00:00,  1.80s/it]


Loss: 0.10067165659964085
Accuracy: 97.78000116348267%


100%|██████████| 157/157 [04:35<00:00,  1.75s/it]


Loss: 0.07840796595811844
Accuracy: 99.01000261306763%


100%|██████████| 157/157 [04:35<00:00,  1.75s/it]


Loss: 0.07814882973465137
Accuracy: 96.14999890327454%


In [ ]:
model = ResNet.to(device)
model.load_state_dict(torch.load("model_state_dict.path"))

In [88]:
model.eval()
with torch.no_grad():
    for images, labels in test_dataloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)

    predictions = torch.argmax(outputs, dim=1)
    epoch_correct_prob = torch.mean((predictions == labels).float()).item()

print(f"Loss: {epoch_loss / len(train_sub_dataset)}")
print(f"Accuracy: {epoch_correct_prob * 100}%")